# 07 — Fairness analysis

> **Dữ liệu synthetic:** kết quả chỉ kiểm tra pipeline, không dùng để kết luận fairness hoặc quyết định tín dụng thực tế.

Notebook so sánh model credit đã chọn với `no_sensitive`, loại `age`, `age_bin`, `marital_status`. Biến thể fairness dùng lại hyperparameter đã chọn, không Optuna và không đọc test.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs' / 'config.yaml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'configs' / 'config.yaml').is_file():
    raise FileNotFoundError('Không tìm thấy configs/config.yaml. Hãy chỉnh PROJECT_ROOT.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.fairness import compute_fairness_metrics
from src.models.predict import load_model_bundle
from src.utils.config import load_config
from src.utils.io import read_parquet_safe

## Performance trade-off

Đọc validation metrics đã lưu; cell này không retrain và không dùng test.

In [ ]:
selected_metrics = json.loads((PROJECT_ROOT / 'models/credit_risk/v1.0.0/metrics.json').read_text(encoding='utf-8'))
no_sensitive_metrics = json.loads((PROJECT_ROOT / 'models/credit_risk/experiments/no_sensitive/metrics.json').read_text(encoding='utf-8'))
comparison = pd.DataFrame([
    {'variant': 'selected', **selected_metrics['validation']},
    {'variant': 'no_sensitive', **no_sensitive_metrics['validation']},
]).drop(columns=['confusion_matrix'])
comparison[['variant', 'qwk', 'macro_f1', 'class_1_f1', 'class_2_f1', 'class_3_f1']]

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=str(PROJECT_ROOT / 'reports/fairness_report.html'), width='100%', height=700))

## Audit lại model được chọn theo nhóm

Cell dưới tái tạo nhãn nhóm từ validation thô để gọi trực tiếp `compute_fairness_metrics`; không refit model.

In [ ]:
from src.data.clean import add_missing_flags, apply_entry_error_rules, handle_outliers
from src.data.load import load_raw
from src.data.split import make_splits
from src.data.validate import load_data_quality_config
from src.features.build_features import build_stateless_features

config = load_config(PROJECT_ROOT / 'configs/config.yaml')
feature_dir = PROJECT_ROOT / config['features']['output_dir']
X_valid = read_parquet_safe(feature_dir / 'X_valid.parquet')
y_valid = read_parquet_safe(feature_dir / 'y_credit_valid.parquet').iloc[:, 0]
quality = load_data_quality_config(PROJECT_ROOT / config['data_quality']['config_path'])
raw = load_raw(config)
cleaned = add_missing_flags(handle_outliers(apply_entry_error_rules(raw, quality), quality))
_, valid_raw, _ = make_splits(cleaned, config)
features = config['features']
experiment = features.get('credit_risk_experiment', {})
valid_features = build_stateless_features(valid_raw, reference_date=features['reference_date'], age_bins=features['age_bins'], income_bins=features['income_bins'], credit_risk_experiment_enabled=bool(experiment.get('enabled', False)))
sensitive = valid_features[config['governance']['sensitive_columns']].reset_index(drop=True)
bundle = load_model_bundle('v1.0.0', 'credit_risk', PROJECT_ROOT / 'models')
output = np.asarray(bundle.model.predict(X_valid.to_numpy(dtype=float)))
predictions = output.argmax(axis=1) if output.ndim == 2 else output.astype(int)
by_group, disparity = compute_fairness_metrics(y_valid, predictions, sensitive, high_risk_class=config['governance']['credit_high_risk_class'])
display(disparity.sort_values(['sensitive_feature', 'metric']))
by_group.sort_values(['sensitive_feature', 'group'])

## Regenerate artifact

Chỉ chạy khi cần tạo lại bundle/report (sẽ refit `no_sensitive` một lần):

```powershell
python -m scripts.run_fairness_report
```